# Inspect Generation Samples

Use this notebook to inspect one fixed `prompt_idx` across baseline, Value-Based edits, and residual-attention edits. It reads the JSONL files under `representation-analysis/outputs/generate` and prints matched excerpts for appendix selection.

In [ ]:
from pathlib import Path
import json
import re
from IPython.display import Markdown, display

# Edit these values.
PROMPT_IDX = 8
MAX_CHARS = 420
KEEP_THINK_TAGS = False
INCLUDE_MLP_BOTH = False

# This notebook lives in _NeurIPS_2026_/results/generate.
ROOT = Path("../../representation-analysis/outputs/generate").resolve()
ROOT

In [ ]:
PARA_SETTINGS = ["-1", "0", "1", "2"]
PERP_SETTINGS = ["0.5", "1.0", "1.5"]

def load_prompt_row(path: Path, prompt_idx: int):
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            if row.get("prompt_idx") == prompt_idx:
                return row
    raise KeyError(f"prompt_idx={prompt_idx} not found in {path}")

def clean_text(text: str, keep_think_tags: bool = False) -> str:
    if not keep_think_tags:
        text = text.replace("<think>", "").replace("</think>", "")
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def maybe_clip(text: str, max_chars: int = MAX_CHARS) -> str:
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 4].rstrip() + " ..."

def get_nested(row, *keys):
    value = row["generations"]
    for key in keys:
        value = value[key]
    return value

def md_escape(text: str) -> str:
    return text.replace("|", "\\|")

def make_record(setting: str, row, *keys):
    text = get_nested(row, *keys)
    text = clean_text(text, keep_think_tags=KEEP_THINK_TAGS)
    return {"setting": setting, "output": maybe_clip(text)}

In [ ]:
baseline_path = ROOT / "para" / "generate_para_scale_1.jsonl"
baseline_row = load_prompt_row(baseline_path, PROMPT_IDX)
prompt = clean_text(baseline_row["prompt"], keep_think_tags=True)

records = []
records.append(make_record("Baseline, no edit", baseline_row, "residual_output", "none"))

for scale in PARA_SETTINGS:
    row = load_prompt_row(ROOT / "para" / f"generate_para_scale_{scale}.jsonl", PROMPT_IDX)
    records.append(make_record(f"Value-Based, s_parallel={scale}", row, "xsa_middle_multihead", "attn"))
    records.append(make_record(f"Residual(attn), s_parallel={scale}", row, "residual_output", "attn"))
    if INCLUDE_MLP_BOTH:
        records.append(make_record(f"Residual(MLP), s_parallel={scale}", row, "residual_output", "mlp"))
        records.append(make_record(f"Residual(both), s_parallel={scale}", row, "residual_output", "both"))

for scale in PERP_SETTINGS:
    row = load_prompt_row(ROOT / "perp" / f"generate_para_scale_1.0_perp_scale_{scale}.jsonl", PROMPT_IDX)
    records.append(make_record(f"Value-Based, s_parallel=1.0, s_perp={scale}", row, "xsa_middle_multihead", "attn"))
    records.append(make_record(f"Residual(attn), s_parallel=1.0, s_perp={scale}", row, "residual_output", "attn"))
    if INCLUDE_MLP_BOTH:
        records.append(make_record(f"Residual(MLP), s_parallel=1.0, s_perp={scale}", row, "residual_output", "mlp"))
        records.append(make_record(f"Residual(both), s_parallel=1.0, s_perp={scale}", row, "residual_output", "both"))

display(Markdown(f"## Prompt {PROMPT_IDX}\n\n> {prompt}"))

table = ["| Setting | Output excerpt |", "|---|---|"]
for record in records:
    table.append(f"| {md_escape(record['setting'])} | {md_escape(record['output'])} |")
display(Markdown("\n".join(table)))

## LaTeX Selection Helper

After choosing rows for the appendix, copy the selected excerpts manually into the LaTeX table. Keep excerpts short so the table remains readable.